In [15]:
import numpy as np
import matplotlib.pyplot as plt
from ngsolve import *
from ngsolve.webgui import Draw

In [16]:
import importlib
import sys
from pathlib import Path
import os 
sys.path.append(str(Path(os.getcwd()).parent / "src"))

import catch_bond_fem
importlib.reload(catch_bond_fem)

import utils 
importlib.reload(utils)

from ngsolve.webgui import Draw

In [17]:

left_edge = 0.5 - 0.05
right_edge = 0.5 + 0.05
tanh = lambda arg: (exp(arg) - exp(-arg))/(exp(arg)+exp(-arg)) 
left_transition = lambda left_edge: 0.5 * (tanh((x - left_edge)/0.01))
right_transition = lambda right_edge: 0.5 * (tanh((right_edge - x)/0.01))
source =  1*(left_transition(left_edge) + right_transition(right_edge)) 


source_sharp = IfPos(x - left_edge, IfPos(right_edge - x, 1.0, 0.0), 0.0)
source_narrow = IfPos(x - (0.5-0.03), IfPos((0.5+0.03) - x, 1.0, 0.0), 0.0)


left_edge = 0.5 - 0.1
right_edge = 0.5 + 0.1
source_wide = 1*(left_transition(left_edge) + right_transition(right_edge)) 

T = 20
tau = 0.01
beta2 = 1
beta1 = 1
eta = 1 


rho_cable = 1
rho_bulk = 1
chi0 = 0.05
chi1 = 1

alpha = 1*source_sharp
rate_ratio = 1 + 9*alpha 

sim = catch_bond_fem.CatchBond1D(
        width=1, maxh=0.01,
        gamma=eta, eta_1=eta*(100*source_narrow+1), eta_2=0,
        k=0.1*rate_ratio, D=1e-4,
        kappa=1e-4,
        beta1=beta1, beta2=beta2, 
        chi0=chi0*alpha,  chi1=chi1*alpha, 
        rho0=lambda t: (10*rho_cable*source_sharp + rho_bulk*(1-source_sharp))/rate_ratio, Qsq =lambda t: -1+2*source_sharp, 
        )


for i in range(len(sim.density.vec.data)): 
    sim.density.vec.data[i] = np.random.normal(1, 0.01)
    sim.nematic_xx.vec.data[i] = np.random.normal(0, 0.01)

sim.simulate(
    tend=T,
    tau=tau,
    save_interval=50 # save once a second 
)


 47%|████▋     | 935/2000 [00:00<00:00, 4734.57it/s]

craete bilinearformapplication


100%|██████████| 2000/2000 [00:00<00:00, 3199.70it/s]


In [ ]:
x_array = np.arange(0, 1, 0.01) 

to_eval = (rho_cable*source_sharp*10 + rho_bulk*(1-source_sharp))/rate_ratio

vals = [] 
for x_point in x_array: 
    vals.append(to_eval(sim.mesh(x_point)))

plt.figure(figsize=(6,4))
plt.rcParams.update({'font.size': 20})
plt.plot(vals)
plt.show() 

In [ ]:
data = sim.export_to_npy(100)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

plt.rcParams.update({'font.size': 15})

im0 = axes[0].imshow(data[:, :, 1], aspect='auto')
axes[0].set_title('v')
axes[0].set_yticks([])
fig.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(data[:, :, 0], aspect='auto')
axes[1].set_title('rho')  
fig.colorbar(im1, ax=axes[1])

im2 = axes[2].imshow(data[:, :, -1], aspect='auto')
axes[2].set_title('Q')
fig.colorbar(im2, ax=axes[2])


plt.show()

In [ ]:
t = -1 
X = np.linspace(0, 1, 100)

rho, v, Q = data[t].T 
dxdv = np.gradient(v, (0.01))
rho_diff = rho - 1 
plt.plot(X, rho_diff, label='rho-1')
plt.plot(X, Q, label='Q')
plt.plot(X, v, label='v')
plt.plot(X, dxdv, label='dxdv')
plt.plot(X, 2*rho*(chi0 + chi1*Q)/(rho+1), label='tension')
plt.axvline(x=right_edge, color='k', linestyle='--')
plt.axvline(x=left_edge, color='k', linestyle='--')  
plt.legend() 
plt.show() 

plt.plot(X, v/np.max(v), label='v') 
plt.plot(X, rho_diff/np.max(np.abs(rho_diff)), label='rho-1')
plt.axvline(x=right_edge, color='k', linestyle='--')
plt.axvline(x=left_edge, color='k', linestyle='--')  
plt.legend()
plt.show() 

In [ ]:
rho = data[:, :, 0] 
v = data[:, :, 1]
Q = data[:, :, 2]

dxdv = np.gradient(v, (0.01), axis=-1) 

T = len(dxdv)
colors = plt.cm.plasma(np.linspace(0, 1, T))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i in range(T): 
    axes[0].plot(dxdv[i], color=colors[i])

    tension = -rho[i]*(chi0+chi1*Q[i])
    axes[1].plot(tension-np.mean(tension), color=colors[i])
    axes[2].plot(rho[i], color=colors[i])
plt.show() 

In [ ]:
from sklearn.linear_model import LinearRegression

A = (rho - np.mean(rho, axis=-1)[:, np.newaxis]).flatten()
B = (Q*rho - np.mean(Q*rho, axis=-1)[:, np.newaxis]).flatten()
C = np.column_stack((A, B))
f = dxdv.flatten() 

model = LinearRegression(fit_intercept=False)  # No intercept as our function doesn't have one
model.fit(C, f)
a, b= model.coef_ 

print(model.coef_)
print(chi0, chi1)

plt.scatter(a*A + b*B, f)
plt.show() 

plt.scatter(A, B, c=f)
